# Order Payments table: cleaning

Run from the folder that contains raw/ and cleaned/ (your VS Code workspace root).
Needs cleaned/orders_clean.csv and cleaned/order_items_clean.csv from the earlier notebooks.

Decisions and findings for this table are written up in `cleaning_notes/README_order_payments.md`.

### 1. Imports and settings

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

### 2. Load the raw payments file

In [ ]:
order_payments = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')

### 3. First inspection: structure, nulls, preview, payment methods

In [ ]:
order_payments.info()
print(order_payments.isna().sum())                  # expect all zeros
print(order_payments.head())
print(order_payments['payment_type'].value_counts())
# credit_card 76,795 / boleto 19,784 / voucher 5,775 / debit_card 1,529 / not_defined 3

### 4. Key check: one row per payment, so the key is the PAIR (order_id, payment_sequential)

In [ ]:
print(order_payments.duplicated(subset=['order_id', 'payment_sequential']).sum())   # expect 0
print(order_payments['order_id'].nunique())                  # 99,440 orders have payments
print(order_payments['payment_installments'].max())          # 24

# (order_id, payment_type) repeats 2,200 times. That is NOT a duplicate problem:
# an order can legitimately be paid twice with the same method (two vouchers, split card payment).
print(order_payments.duplicated(subset=['order_id', 'payment_type']).sum())

### 5. Numeric profile and the odd rows

In [ ]:
print(order_payments[['payment_installments', 'payment_value']].describe())

print(order_payments[order_payments['payment_value'] <= 0])            # 9 rows: 6 voucher + 3 not_defined
print(order_payments[order_payments['payment_installments'] < 1])      # 2 credit_card rows with 0 installments
print(order_payments[order_payments['payment_type'] == 'not_defined']) # 3 rows, all value 0.0

### 6. Join integrity with orders

In [ ]:
orders = pd.read_csv(CLEAN / 'orders_clean.csv', usecols=['order_id', 'order_status'])

print(order_payments['order_id'].isin(orders['order_id']).value_counts())   # True 103,886 (no orphan payments)
no_pay = ~orders['order_id'].isin(order_payments['order_id'])
print(orders.loc[no_pay])                    # 1 order, status delivered (a source-data gap)

nd = order_payments.loc[order_payments['payment_type'] == 'not_defined', 'order_id']
print(orders[orders['order_id'].isin(nd)])   # all 3 are canceled

### 7. Flag problem rows (kept, not dropped)

In [ ]:
order_payments['has_data_issue'] = (
    (order_payments['payment_installments'] < 1) | (order_payments['payment_value'] == 0)
)
print(order_payments['has_data_issue'].value_counts())      # False 103,875 / True 11

### 8. Reconcile payments against item values, per order

In [ ]:
items = pd.read_csv(CLEAN / 'order_items_clean.csv', usecols=['order_id', 'item_total'])

items_total = items.groupby('order_id')['item_total'].sum().rename('items_total')
paid_total = order_payments.groupby('order_id')['payment_value'].sum().rename('paid_total')

rec = pd.concat([items_total, paid_total], axis=1)
print(rec.isna().sum())          # items_total 775 (orders without items) / paid_total 1 (the unpaid order)

both = rec.dropna().copy()
both['diff'] = (both['paid_total'] - both['items_total']).round(2)
print(len(both))                              # 98,665
print((both['diff'].abs() > 0.01).sum())      # 303 mismatches (0.3%)
print(both['diff'].describe())                # min -51.62 / max 182.81 / median 0

print(rec.loc[rec['items_total'].isna(), 'paid_total'].sum().round(2))   # 162,591.95 paid on orders with no items

# Payment types of the 303 mismatched orders
pt = order_payments.groupby('order_id')['payment_type'].agg(lambda s: '+'.join(sorted(s.unique())))
mism = both[both['diff'].abs() > 0.01]
print(mism.join(pt)['payment_type'].value_counts())
# credit_card 275 / boleto 14 / credit_card+voucher 7 / debit_card 7

### 9. Save the cleaned table and verify the file on disk

In [ ]:
order_payments.to_csv(CLEAN / 'order_payments_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'order_payments_clean.csv')
print(check.shape)                                                          # expect (103886, 6)
print(check.duplicated(subset=['order_id', 'payment_sequential']).sum())    # expect 0
print(check['has_data_issue'].value_counts())                               # True 11 / False 103,875
print(check['payment_value'].sum().round(2))                                # 16,008,872.12